# Medication Companion — Kaggle Demo

Multi-agent prescription analysis for patients in India.
Runs all **5 agents** with `InMemorySessionService` — no GCP credentials required.

**Kaggle setup:** Add a notebook secret named `GEMINI_API_KEY` (or `GOOGLE_API_KEY`). Attach this repo as a Kaggle dataset, or clone from GitHub and set `REPO_ROOT` in the setup cell.

> **Disclaimer:** This is an educational prototype, not a medical device. It is not a substitute for pharmacist or doctor advice. 

## 1. Setup

Set `GEMINI_API_KEY` (or `GOOGLE_API_KEY`) to run the full LLM pipeline. Memory and session demos below work without any API key.

In [ ]:
import os
import sys
from pathlib import Path

# Locate repo root whether the notebook is opened from /notebooks or repo root
cwd = Path.cwd()
if (cwd / "backend").is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / "backend").is_dir():
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd

BACKEND = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND))
os.chdir(BACKEND)

os.environ.setdefault("MEMORY_BACKEND", "local")
os.environ.setdefault("ENVIRONMENT", "local")
os.environ.setdefault("DEV_PATIENT_ID", "demo-patient-001")

print(f"Repo root: {REPO_ROOT}")
print(f"MEMORY_BACKEND={os.environ['MEMORY_BACKEND']}")

In [ ]:
# Install dependencies when running on Kaggle (skip if already installed)
try:
    import google.adk  # noqa: F401
except ImportError:
    %pip install -q -r requirements.txt

## 2. In-memory session & memory services

Day 3 (Context Engineering): short-term session state + long-term cross-visit memory.

In [ ]:
from google.adk.sessions import InMemorySessionService
from memory.session_service import create_session_service
from memory.memory_service import create_memory_service

session_service = create_session_service()
memory_service = create_memory_service()

assert isinstance(session_service, InMemorySessionService)
assert memory_service.is_local()
print("✓ InMemorySessionService ready")
print("✓ In-process memory store ready")

In [ ]:
import asyncio

PATIENT_ID = os.environ["DEV_PATIENT_ID"]

async def demo_cross_visit_memory():
  # Visit 1: patient was prescribed warfarin
  await memory_service.save_visit(
      patient_id=PATIENT_ID,
      resolved_drug_names=["warfarin"],
      severity="INFO",
  )
  # Visit 2: new prescription includes aspirin (known interaction with warfarin)
  history = await memory_service.get_medications_for_patient(PATIENT_ID)
  print(f"Prior visits for {PATIENT_ID}: {len(history)}")
  for visit in history:
      print(f"  {visit['visit_timestamp'][:10]} — {visit['resolved_drugs']} ({visit['severity_summary']})")

await demo_cross_visit_memory()

## 3. Build the 5-agent pipeline

| Agent | Role | Course day |
|-------|------|------------|
| 1 Reader | OCR + Gate 1 confidence check | Day 1 |
| 2 Resolver | Brand → generic, FDC split (FunctionTools) | Day 2 |
| 3 Safety | Cross-visit interaction check (memory) | Day 3 |
| 4 Education | Plain-language explanation | Day 1 |
| 5 Localisation | Translate + TTS (in-process SequentialAgent step) | Day 2 |

Production orchestration: `backend/agent.py` — `SequentialAgent` with policy callbacks and Memory Bank persistence.

In [ ]:
from google.adk.agents import SequentialAgent
from agents.agent1_reader import create_reader_agent
from agents.agent2_resolver import create_resolver_agent
from agents.agent3_safety import create_safety_agent
from agents.agent4_education import create_education_agent
from agents.agent5_localisation import create_localisation_agent

reader_agent = create_reader_agent()
resolver_agent = create_resolver_agent()
safety_agent = create_safety_agent(memory_service=memory_service)
education_agent = create_education_agent()
localisation_agent = create_localisation_agent()

root_agent = SequentialAgent(
    name="medication_companion",
    sub_agents=[
        reader_agent,
        resolver_agent,
        safety_agent,
        education_agent,
        localisation_agent,
    ],
)

agents = {
    "1_reader": reader_agent,
    "2_resolver": resolver_agent,
    "3_safety": safety_agent,
    "4_education": education_agent,
    "5_localisation": localisation_agent,
}
for name, agent in agents.items():
    print(f"✓ {name}: {agent.name}")

## 4. Tool demo (Agent 2) — no LLM required

Indian brand lookup from bundled CSV + FDC splitting.

In [ ]:
from tools.drug_lookup import drug_lookup
from tools.combo_splitter import combo_splitter

for brand in ["Azee", "Augmentin", "Pantocid DSR"]:
    result = drug_lookup(brand)
    print(f"{brand:15} → {result}")
    if result.get("is_combo"):
        print(f"{'':15}   components: {combo_splitter(brand)}")

## 5. Run the full pipeline (requires Gemini API key)

Provide a prescription image path or use the sample text prompt below.

In [ ]:
from google.adk.runners import Runner
from google.genai import types

api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not api_key:
    print("Set GEMINI_API_KEY to run the full agent pipeline.")
    print("Sections 2–4 above work without an API key.")
else:
    runner = Runner(
        agent=root_agent,
        app_name="medication_companion",
        session_service=session_service,
    )

    session = await session_service.create_session(
        app_name="medication_companion",
        user_id=PATIENT_ID,
    )

    # Text-only demo when no prescription image is available
    message = types.Content(
        role="user",
        parts=[types.Part(text=(
            "Prescription drugs (as written on the slip): Azee 500, Ecosprin 75. "
            "Resolve, check interactions against my history, and explain findings."
        ))],
    )

    print(f"Running pipeline for patient {PATIENT_ID} (session {session.id})...\n")
    async for event in runner.run_async(
        user_id=PATIENT_ID,
        session_id=session.id,
        new_message=message,
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text[:2000])

## 6. Agent 5 — Localisation + audio

Agent 5 is the final step of the production `SequentialAgent` (`backend/agents/agent5_localisation.py`). Run this cell to translate the education summary into Hindi.

In [ ]:
if not api_key:
    print("Set GEMINI_API_KEY to run Agent 5 localisation.")
else:
    loc_runner = Runner(
        agent=localisation_agent,
        app_name="medication_companion",
        session_service=session_service,
    )
    loc_session = await session_service.create_session(
        app_name="medication_companion",
        user_id=PATIENT_ID,
    )
    english_summary = (
        "Ecosprin (aspirin) may interact with warfarin from your prior visit. "
        "This combination can increase bleeding risk. "
        "Please discuss this with your doctor or pharmacist before making any changes."
    )
    loc_message = types.Content(
        role="user",
        parts=[types.Part(text=f"Target language: hi-IN\n\n{english_summary}")],
    )
    async for event in loc_runner.run_async(
        user_id=PATIENT_ID,
        session_id=loc_session.id,
        new_message=loc_message,
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)